# Market Microstructure: Order Books, Liquidity & Execution

**Level:** Intermediate  
**Tags:** Microstructure, Order Book, Liquidity, Bid-Ask Spread, Market Impact  
**Estimated Time:** 120-150 minutes

## Overview

Deep dive into market microstructure - the mechanics of how markets actually work. Learn about order books, liquidity metrics, market impact, and optimal execution strategies.

## Learning Objectives

✓ Understand limit order book dynamics  
✓ Calculate bid-ask spreads and liquidity metrics  
✓ Measure market impact and price slippage  
✓ Analyze order flow and trade classification  
✓ Implement Kyle's Lambda and Amihud illiquidity  
✓ Build optimal execution models (VWAP, TWAP)  
✓ Detect toxic flow and adverse selection  

## Prerequisites

- Understanding of market orders vs limit orders
- Basic statistics and probability
- Python programming (pandas, numpy)
- Time series data handling

## References

1. O'Hara, M. (1995). *Market Microstructure Theory*. Blackwell.
2. Hasbrouck, J. (2007). *Empirical Market Microstructure*. Oxford University Press.
3. Almgren, R., & Chriss, N. (2001). "Optimal Execution of Portfolio Transactions". *Journal of Risk*.
4. Kyle, A.S. (1985). "Continuous Auctions and Insider Trading". *Econometrica*.
5. Amihud, Y. (2002). "Illiquidity and Stock Returns". *Journal of Financial Markets*.

In [ ]:
# Import libraries\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom scipy import stats\nfrom scipy.optimize import minimize\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configuration\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\n%matplotlib inline\nnp.random.seed(42)\n\nprint('Libraries imported successfully!')

## 1. Introduction to Market Microstructure

### What is Market Microstructure?

Market microstructure studies the **process by which latent demands are ultimately translated into prices and volumes**. It examines:

- How orders are submitted and matched
- Price formation mechanisms  
- Information asymmetry and adverse selection
- Transaction costs and market impact
- Liquidity provision and consumption

### Why It Matters

**For Traders:**
- Minimize execution costs
- Optimize order routing
- Detect liquidity opportunities

**For Market Makers:**
- Set optimal bid-ask spreads
- Manage inventory risk
- Avoid adverse selection

**For Regulators:**
- Monitor market quality
- Detect manipulation
- Assess fairness and efficiency

### Key Concepts

**1. Limit Order Book (LOB)**
```
           BID                    ASK
Price  |  Size       Price  |  Size
-------|-------      -------|-------
99.98  |  500        100.02 |  300
99.97  |  800        100.03 |  600
99.96  |  1200       100.04 |  400
```

**2. Bid-Ask Spread**
- **Quoted Spread**: Ask - Bid
- **Effective Spread**: 2 × |Trade Price - Midpoint|
- **Realized Spread**: Measures post-trade price movement

**3. Market Impact**
- **Temporary**: Price moves during execution, then reverts
- **Permanent**: Information content causes persistent price change

**4. Liquidity Metrics**
- **Depth**: Volume available at best prices
- **Resilience**: Speed of order book recovery
- **Tightness**: Bid-ask spread size

### Historical Evolution

- **Pre-2000**: Floor trading, specialists, manual markets
- **2000-2007**: Electronic trading proliferates
- **2007-2010**: High-frequency trading emerges
- **2010+**: Fragmentation, dark pools, maker-taker pricing
- **2020+**: Retail order flow payment, meme stocks

In [ ]:
# Simulate a realistic limit order book
np.random.seed(42)

class LimitOrderBook:
    """Simplified limit order book simulator"""
    
    def __init__(self, mid_price=100, tick_size=0.01, depth_levels=10):
        self.mid_price = mid_price
        self.tick_size = tick_size
        self.depth_levels = depth_levels
        self.generate_book()
    
    def generate_book(self):
        """Generate realistic order book with depth"""
        # Bid side (below mid)
        self.bid_prices = np.array([self.mid_price - (i+1)*self.tick_size 
                                     for i in range(self.depth_levels)])
        # Ask side (above mid)
        self.ask_prices = np.array([self.mid_price + (i+1)*self.tick_size 
                                     for i in range(self.depth_levels)])
        
        # Size decays with distance from mid (exponential)
        base_size = 1000
        decay_rate = 0.8
        self.bid_sizes = np.array([int(base_size * (decay_rate ** i)) 
                                    for i in range(self.depth_levels)])
        self.ask_sizes = np.array([int(base_size * (decay_rate ** i)) 
                                    for i in range(self.depth_levels)])
    
    def get_spread(self):
        """Calculate bid-ask spread"""
        return self.ask_prices[0] - self.bid_prices[0]
    
    def get_mid_price(self):
        """Calculate mid-price"""
        return (self.bid_prices[0] + self.ask_prices[0]) / 2
    
    def display(self):
        """Display order book"""
        print("=" * 70)
        print(f"{'LIMIT ORDER BOOK':^70}")
        print("=" * 70)
        print(f"{'BID SIDE':^30} | {'ASK SIDE':^30}")
        print(f"{'Price':>12} {'Size':>12} {'Depth':>6} | {'Price':>12} {'Size':>12} {'Depth':>6}")
        print("-" * 70)
        
        cum_bid = 0
        cum_ask = 0
        for i in range(self.depth_levels):
            cum_bid += self.bid_sizes[i]
            cum_ask += self.ask_sizes[i]
            print(f"{self.bid_prices[i]:12.2f} {self.bid_sizes[i]:12,} {cum_bid:6,} | "
                  f"{self.ask_prices[i]:12.2f} {self.ask_sizes[i]:12,} {cum_ask:6,}")
        
        print("-" * 70)
        print(f"Best Bid: ${self.bid_prices[0]:.2f} x {self.bid_sizes[0]:,}")
        print(f"Best Ask: ${self.ask_prices[0]:.2f} x {self.ask_sizes[0]:,}")
        print(f"Spread: ${self.get_spread():.2f} ({self.get_spread()/self.mid_price*10000:.1f} bps)")
        print(f"Mid-Price: ${self.get_mid_price():.2f}")
        print("=" * 70)

# Create and display order book
lob = LimitOrderBook(mid_price=100, tick_size=0.01, depth_levels=10)
lob.display()

# Visualize the order book
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Order book shape
ax1.barh(lob.bid_prices, lob.bid_sizes, height=0.008, 
         color='green', alpha=0.7, label='Bids')
ax1.barh(lob.ask_prices, -lob.ask_sizes, height=0.008, 
         color='red', alpha=0.7, label='Asks')
ax1.axhline(y=lob.get_mid_price(), color='black', linestyle='--', 
            linewidth=2, label='Mid-Price')
ax1.set_xlabel('Size (shares)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
ax1.set_title('Limit Order Book Depth', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='x')

# Cumulative depth
cum_bid = np.cumsum(lob.bid_sizes)
cum_ask = np.cumsum(lob.ask_sizes)

ax2.plot(cum_bid, lob.bid_prices, 'o-', linewidth=2.5, 
         color='green', markersize=8, label='Bid Depth')
ax2.plot(cum_ask, lob.ask_prices, 'o-', linewidth=2.5, 
         color='red', markersize=8, label='Ask Depth')
ax2.axhline(y=lob.get_mid_price(), color='black', linestyle='--', 
            linewidth=2, label='Mid-Price')
ax2.set_xlabel('Cumulative Size (shares)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
ax2.set_title('Cumulative Order Book Depth', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Mathematical Foundation\n\n$$\\text{Equation here}$$

## 3. Python Implementation

## 4. Visualization and Analysis

## 5. Practical Applications

## Summary\n\nThis notebook covered market microstructure diagnostics. Key takeaways:\n\n- Practical implementation with Python\n- Visualizations and interpretations\n- Real-world applications\n- Best practices and pitfalls\n\n### Next Steps\n\n- Practice with real market data\n- Explore related topics\n- Build your own variations\n\n### Additional Resources\n\n- [QuantEdX Courses](https://www.quantedx.com/courses)\n- [Community Forum](https://www.quantedx.com/community)\n- [GitHub Repository](https://github.com/quantedx)